### Choose where this notebook runs

This is the **IQM edition**: saving and loading circuits is backend-independent, so the lesson is
unchanged; one extra cell runs a circuit loaded from disk on **Garnet** (20 qubits) or **Emerald**
(54 qubits) via IQM Resonance, after transpilation to the IQM native gate set **PRX + CZ**. Pick the backend
by editing the single `TARGET` line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |


# Saving Circuits to Disk

**Course**: Quantum Computing Laboratory  
**Version**: 2.2 (Qiskit 2.2 Compatible)  
**Last Updated**: November 2025

---

In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from iqm_backend import get_backend, run_counts` | the helper module of this folder: `run_counts(circuit, TARGET, shots=...)` = `transpile_to_IQM` (rewrite the circuit with the native `r` = PRX rotations and `cz` gates, routed on the chip's square lattice) → `backend.run` → counts as a plain dict; `run_counts(..., return_transpiled=True)` also returns the transpiled circuit |
| `TARGET = "fake_apollo"` | a local noisy simulator with the 20-qubit Garnet topology: no token, no credits, realistic gate and readout noise; `"garnet:mock"` checks the cloud pipeline but returns RANDOM bitstrings; `"garnet"` / `"emerald"` are the real QPUs (token + credits; IQM Resonance declines circuits above 10 000 native instructions) |
| `get_backend(TARGET)` | the raw Qiskit backend object (name, qubit count, coupling map) if you want to call `backend.run` yourself |

</details>


## 📚 Overview

Learn how to save and load quantum circuits using **QPY (Qiskit Python)** format - the standard serialization format for Qiskit circuits.

### Topics Covered:
- Saving single circuits
- Loading circuits from disk
- Saving multiple circuits
- Version compatibility
- Best practices for circuit persistence

## 🎯 Learning Objectives

1. ✅ Understand QPY format
2. ✅ Save circuits to files
3. ✅ Load circuits from files
4. ✅ Manage multiple circuits
5. ✅ Apply to version control and collaboration

## 🔧 Prerequisites

**Required Knowledge:**
- Basic circuit construction
- Python file I/O basics

**Required Software:**
- Python 3.9+
- Qiskit 2.2+

**Recommended Previous Notebooks:**
- `Circuit_Construction_v2_2_enhanced.ipynb`

## 📖 Why Save Circuits?

### Use Cases

**1. Version Control**
- Track circuit evolution over time
- Collaborate with Git
- Reproducible research

**2. Circuit Libraries**
- Build reusable circuit collections
- Share with collaborators
- Standard benchmarks

**3. Long Experiments**
- Save intermediate results
- Resume interrupted work
- Checkpoint complex workflows

**4. Production Deployment**
- Deploy to different environments
- Separate development from execution
- Maintain circuit consistency

### QPY Format Advantages

✅ **Binary format** - Efficient storage  
✅ **Version resilient** - Forward/backward compatibility  
✅ **Complete preservation** - All circuit metadata  
✅ **Fast I/O** - Quick serialization/deserialization  
✅ **Cross-platform** - Works everywhere  

In [ ]:
# Qiskit 2.2 imports
from qiskit import QuantumCircuit, qpy                  # qpy = Qiskit's binary serialization module: qpy.dump / qpy.load
from qiskit.circuit.library import QFT, GroverOperator  # ready-made circuits from the library (QFT is used below)
from qiskit.visualization import circuit_drawer         # the function behind qc.draw(); imported for completeness
import os                                               # file system: paths, sizes, makedirs, remove
import glob                                             # wildcard file search: glob.glob('dir/*.qpy')
import numpy as np                                      # arccos, pi (used in the In-depth analysis)
from IPython.display import display                      # show a figure (e.g. a circuit drawing) from the middle of a cell
def draw(qc, fold=-1, **kw): return qc.draw("mpl", style="clifford", fold=fold, **kw)   # every circuit picture: matplotlib drawer; fold=-1 = one long row, fold=N wraps after N columns

print("✓ All imports successful!")
print("  Qiskit 2.2 QPY module loaded")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `from qiskit import qpy` | QPY is Qiskit's own binary format for circuits: `qpy.dump(circuit_or_list, file)` writes, `qpy.load(file)` reads back a *list* of circuits. Unlike `pickle` it is versioned and readable by future Qiskit releases |
| `os`, `glob` | standard-library file handling: `os.path.getsize`, `os.makedirs`, `os.remove`, and `glob.glob(pattern)` to list files matching a wildcard |
| `draw(qc)` | our helper around `qc.draw("mpl", style="clifford")`; `display(...)` shows the picture from anywhere in a cell |

</details>


## 💾 Saving a Single Circuit

### Basic Save Operation

In [ ]:
# Create a simple Bell state circuit
qc = QuantumCircuit(2, 2, name='Bell_State')   # 2 qubits, 2 classical bits; name= is stored in the file and shown in drawings
qc.h(0)                                        # Hadamard on qubit 0
qc.cx(0, 1)                                    # CNOT 0 -> 1: Bell state (|00> + |11>)/sqrt2
qc.measure_all()                               # adds its own 2-bit register 'meas' and measures every qubit

print("Circuit created:")
print(f"  Name: {qc.name}")
print(f"  Qubits: {qc.num_qubits}")
print(f"  Depth: {qc.depth()}")                # longest path of gates (barrier and measurements included)
print(f"  Gates: {qc.size()}")                 # total number of instructions

# Display circuit
display(draw(qc)); print("↑ the circuit we are about to save: H, CNOT, then the barrier + measurements added by measure_all()")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `QuantumCircuit(2, 2, name='Bell_State')` | the name travels with the circuit into the QPY file — give circuits meaningful names before saving them |
| `qc.measure_all()` | adds a *new* classical register `meas` (the two bits declared in the constructor stay unused) — that is why the drawing shows two classical wires `c` and `meas` |
| `qc.depth()`, `qc.size()` | cheap fingerprints of a circuit (here 3 and 4): we print them again after loading to check that nothing was lost |

</details>


In [ ]:
# Save circuit to file
filename = 'bell_state.qpy'

with open(filename, 'wb') as file:             # 'wb' = write, binary: QPY is a binary format
    qpy.dump(qc, file)                         # serialize the circuit into the open file (a list of circuits works too)

print(f"✓ Circuit saved to: {filename}")
print(f"  File size: {os.path.getsize(filename)} bytes")   # a few hundred bytes for a 2-qubit circuit


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `with open(filename, 'wb') as file:` | a context manager: the file is closed automatically at the end of the block, even if an error occurs; `'wb'` because QPY is binary (never `'w'`) |
| `qpy.dump(qc, file)` | writes the circuit — gates, registers, name, metadata, parameters — not just the drawing. The same call accepts a list of circuits |
| `os.path.getsize(filename)` | size in bytes: QPY stores one record per instruction, so the file grows linearly with the circuit |

</details>


## 📂 Loading a Circuit

### Basic Load Operation

In [ ]:
# Load circuit from file
with open(filename, 'rb') as file:             # 'rb' = read, binary
    loaded_circuits = qpy.load(file)           # ALWAYS returns a list, even if one circuit was saved
    loaded_qc = loaded_circuits[0]  # QPY returns a list

print("✓ Circuit loaded successfully")
print(f"  Name: {loaded_qc.name}")
print(f"  Qubits: {loaded_qc.num_qubits}")
print(f"  Depth: {loaded_qc.depth()}")
print(f"  Gates: {loaded_qc.size()}")

# Verify it matches original
print(f"\n✓ Matches original: {loaded_qc == qc}")   # == compares the instruction lists and registers, not the Python object identity

# Display loaded circuit
display(draw(loaded_qc)); print("↑ the circuit read back from bell_state.qpy: identical to the one above, register names included")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qpy.load(file)` → `[...]` | a list, because a file may hold many circuits; `[0]` picks the first. Forgetting this is the classic QPY mistake |
| `loaded_qc == qc` | `QuantumCircuit.__eq__` compares the content (same gates on the same wires, same registers); `True` proves the round trip was lossless |
| same name, depth, size | the quick checklist after any load: the file carried everything, including the register names shown in the drawing |

</details>


## 📦 Saving Multiple Circuits

### Circuit Collections

In [ ]:
# Create multiple circuits
circuits = []                                  # a plain Python list of QuantumCircuit objects

# 1. Bell state
qc_bell = QuantumCircuit(2, name='Bell')
qc_bell.h(0)
qc_bell.cx(0, 1)
circuits.append(qc_bell)

# 2. GHZ state (3-qubit entanglement)
qc_ghz = QuantumCircuit(3, name='GHZ')
qc_ghz.h(0)
qc_ghz.cx(0, 1)                                # copy the superposition of qubit 0 onto qubit 1 ...
qc_ghz.cx(0, 2)                                # ... and onto qubit 2: (|000> + |111>)/sqrt2
circuits.append(qc_ghz)

# 3. W state (different 3-qubit entanglement)
qc_w = QuantumCircuit(3, name='W_State')
qc_w.ry(1.9106, 0)  # Specific angles for W state   (1.9106 = 2*arccos(1/sqrt3): the first step of a W-state preparation)
qc_w.cx(0, 1)
qc_w.x(0)
qc_w.ry(1.9106, 0)
qc_w.cx(0, 2)
circuits.append(qc_w)

# 4. QFT circuit
qc_qft = QFT(3, name='QFT_3')                  # library circuit: 3-qubit quantum Fourier transform (one opaque block until decomposed)
circuits.append(qc_qft)

print(f"Created {len(circuits)} circuits:")
for i, circ in enumerate(circuits):            # enumerate gives (index, element) pairs
    print(f"  {i+1}. {circ.name}: {circ.num_qubits} qubits, depth {circ.depth()}")

display(draw(qc_w)); print("↑ circuit 3 ('W_State'): two Ry(1.9106) rotations, an X and two CNOTs — a 3-qubit example state (see the In-depth analysis for an exact W state)")
display(draw(qc_qft.decompose())); print(f"↑ circuit 4 ('QFT_3') after decompose(): Hadamards, controlled phases π/2 and π/4, and a final swap — {dict(qc_qft.decompose().count_ops())}, depth {qc_qft.decompose().depth()}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `circuits = []` … `.append(...)` | a Python list is all QPY needs to save a whole library in one file; the order of the list is the order in the file |
| `qc_w` (`ry(1.9106)`, …) | 1.9106 = 2·arccos(1/√3) is the angle that puts probability 1/3 on \|0⟩ of qubit 0 — the first step of a W-state recipe. The remaining gates of this cell do *not* complete it (the exact output has probabilities 2/9, 2/9, 1/9, 4/9 on 000, 010, 101, 111); for the saving/loading lesson any 3-qubit circuit will do, and the In-depth analysis builds a true W state |
| `QFT(3, name='QFT_3')` | a library circuit: `depth()` is 1 because it is a single block; `.decompose()` opens it into H, controlled-phase and swap gates (depth 6) — QPY saves the block *and* its definition |
| `display(draw(...))` | two of the four circuits were never shown: the pictures make the file contents concrete before we serialize them |

</details>


In [ ]:
# Save all circuits to one file
multi_filename = 'circuit_library.qpy'

with open(multi_filename, 'wb') as file:
    qpy.dump(circuits, file)                   # a LIST of circuits -> one file; order preserved

print(f"✓ {len(circuits)} circuits saved to: {multi_filename}")
print(f"  File size: {os.path.getsize(multi_filename)} bytes")   # roughly the sum of the four single-circuit files


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qpy.dump(circuits, file)` | the same function accepts a list: one file, four circuits, order preserved — the simplest way to ship a benchmark set |
| file size | about four times the Bell-state file: QPY is compact (an instruction is a few bytes) and the QFT block carries its definition |

</details>


### Loading Multiple Circuits

In [ ]:
# Load all circuits
with open(multi_filename, 'rb') as file:
    loaded_circuits = qpy.load(file)           # the list comes back in the same order

print(f"✓ Loaded {len(loaded_circuits)} circuits:")
for i, circ in enumerate(loaded_circuits):
    print(f"  {i+1}. {circ.name}: {circ.num_qubits} qubits, depth {circ.depth()}")

# Display first circuit
print(f"\nDisplaying first circuit ({loaded_circuits[0].name}):")
display(draw(loaded_circuits[0])); print("↑ the first circuit of the library file, read back: the 2-qubit Bell preparation (no measurements were saved)")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qpy.load(file)` on a multi-circuit file | returns the whole list; names, qubit counts and depths match the table printed before saving |
| `loaded_circuits[0]` | indexing by position is fragile for a library: the In-depth analysis adds a `load_by_name` helper |

</details>


In [ ]:
# IQM: a circuit loaded from disk runs like any other — read the GHZ circuit back from the library file and run it on TARGET
with open(multi_filename, 'rb') as file:
    ghz_from_disk = qpy.load(file)[1]                 # index 1 = the second circuit that was saved = 'GHZ'
ghz_run = ghz_from_disk.copy()                        # keep the loaded circuit untouched
ghz_run.measure_all()                                 # the stored circuit has no measurements: add them
counts, ghz_iqm = run_counts(ghz_run, TARGET, shots=1024, return_transpiled=True)   # transpile to PRX + CZ, run on TARGET, counts
print(f"{ghz_from_disk.name} from {multi_filename}: {ghz_iqm.size()} native instructions after transpile_to_IQM, "
      f"depth {ghz_iqm.depth()}, gates {dict(ghz_iqm.count_ops())}")   # 'r' = PRX rotation, 'cz' = controlled-Z
print("counts:", dict(sorted(counts.items(), key=lambda kv: -kv[1])))   # ideal: only '000' and '111'; on fake_apollo a few noisy strings appear


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qpy.load(file)[1]` → `.copy()` → `.measure_all()` | a saved circuit is a normal `QuantumCircuit`: copy it, add the measurements the library version did not have, and it is ready for any backend |
| `run_counts(ghz_run, TARGET, shots=1024, return_transpiled=True)` | `transpile_to_IQM` maps H and the two CNOTs onto the native `r` (PRX) and `cz` gates and onto three connected qubits of the square lattice, then `backend.run`; the transpiled circuit is returned for inspection |
| `TARGET` | `fake_apollo` = local noisy simulator (the default); `garnet:mock` = cloud pipeline check with random bitstrings; `garnet` / `emerald` = the real QPUs (token + credits) |
| the counts | a GHZ state gives only `000` and `111`, about half each; the small bars on other strings are the noise of the 20-qubit model — the file format did not change the physics |

</details>


## 🔄 Advanced: Circuit Management

### Organized Storage

In [ ]:
# Create organized directory structure
import os

# Create directories if they don't exist
os.makedirs('circuits/bell_states', exist_ok=True)   # makedirs creates the parents too; exist_ok=True: no error if it is already there
os.makedirs('circuits/algorithms', exist_ok=True)
os.makedirs('circuits/benchmarks', exist_ok=True)

print("✓ Directory structure created:")
print("  circuits/")
print("    ├── bell_states/")
print("    ├── algorithms/")
print("    └── benchmarks/")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `os.makedirs(path, exist_ok=True)` | creates nested folders in one call and is idempotent — safe to re-run the cell |
| `circuits/<category>/` | a convention, not a Qiskit feature: QPY files are ordinary files, so any folder layout (and any version-control tool) works |

</details>


In [ ]:
# Save circuits to organized locations
def save_circuit(circuit, category, filename):
    """Save circuit to organized directory structure."""
    path = f'circuits/{category}/{filename}'       # build the path from the category
    with open(path, 'wb') as file:
        qpy.dump(circuit, file)
    return path                                    # hand the path back so the caller can log it

# Save our circuits
paths = []
paths.append(save_circuit(circuits[0], 'bell_states', 'bell.qpy'))
paths.append(save_circuit(circuits[1], 'bell_states', 'ghz.qpy'))
paths.append(save_circuit(circuits[2], 'bell_states', 'w_state.qpy'))
paths.append(save_circuit(circuits[3], 'algorithms', 'qft_3.qpy'))

print("✓ Circuits saved to organized directories:")
for path in paths:
    print(f"  {path}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `def save_circuit(circuit, category, filename)` | a thin wrapper that fixes the folder convention in one place; returning the path lets the caller keep a manifest |
| one circuit per file | the alternative to the single library file: easier to version and diff individually, slightly more files to manage |

</details>


### Loading from Directory

In [ ]:
# Load all circuits from a directory
def load_circuits_from_directory(directory):
    """Load all QPY files from a directory."""
    circuits = []
    pattern = f'{directory}/*.qpy'                 # wildcard: every .qpy file in that folder

    for filepath in glob.glob(pattern):            # glob expands the wildcard into a list of paths (order not guaranteed)
        with open(filepath, 'rb') as file:
            loaded = qpy.load(file)
            circuits.extend(loaded)                # extend, not append: each file holds a LIST of circuits

    return circuits

# Load all Bell state circuits
bell_circuits = load_circuits_from_directory('circuits/bell_states')

print(f"✓ Loaded {len(bell_circuits)} Bell state circuits:")
for circ in bell_circuits:
    print(f"  - {circ.name}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `glob.glob('circuits/bell_states/*.qpy')` | lists the matching files; the order follows the file system, so never rely on it — use names (or metadata) to identify circuits |
| `circuits.extend(loaded)` | `qpy.load` returns a list even for a single circuit; `extend` flattens the lists, `append` would nest them |

</details>


## 📊 Circuit Metadata

### Preserving Circuit Information

In [ ]:
# Create circuit with metadata
qc_meta = QuantumCircuit(2, name='Documented_Circuit')
qc_meta.h(0)
qc_meta.cx(0, 1)

# Add metadata (stored in circuit)
qc_meta.metadata = {                           # any JSON-serializable dict; it is saved inside the QPY file
    'author': 'Roberto Cappuccio',
    'date': '2025-11-19',
    'purpose': 'Educational example',
    'version': '1.0',
    'notes': 'Standard Bell state preparation'
}

print("Circuit with metadata:")
print(f"  Name: {qc_meta.name}")
print(f"  Metadata: {qc_meta.metadata}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qc_meta.metadata = {...}` | a free-form dictionary attached to the circuit (strings, numbers, lists, nested dicts — anything JSON can hold); Qiskit never reads it, it is for you and your collaborators |
| why | provenance: who built the circuit, for which experiment, with which Qiskit version — the information a bare gate list cannot carry |

</details>


In [ ]:
# Save and load with metadata
with open('circuit_with_metadata.qpy', 'wb') as file:
    qpy.dump(qc_meta, file)

with open('circuit_with_metadata.qpy', 'rb') as file:
    loaded_meta = qpy.load(file)[0]            # [0]: one circuit was saved

print("✓ Loaded circuit with metadata:")
print(f"  Name: {loaded_meta.name}")
print(f"  Metadata preserved: {loaded_meta.metadata}")
print(f"\nMetadata details:")
for key, value in loaded_meta.metadata.items():   # .items() iterates over (key, value) pairs of the dict
    print(f"  {key}: {value}")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `qpy.dump` / `qpy.load` with metadata | the dictionary comes back intact: QPY serializes `metadata` as JSON inside the file |
| `.items()` | the idiomatic way to loop over a dictionary's keys and values at the same time |

</details>


## ⚠️ Best Practices

### 1. File Naming Conventions

In [ ]:
# Good naming practices
def generate_circuit_filename(circuit, version='v1'):
    """
    Generate standardized filename for circuit.

    Format: <name>_<qubits>q_<version>.qpy
    """
    safe_name = circuit.name.lower().replace(' ', '_')   # lowercase, spaces -> underscores: safe on every file system
    return f"{safe_name}_{circuit.num_qubits}q_{version}.qpy"

# Example
qc_example = QuantumCircuit(5, name='Grover Oracle')
filename = generate_circuit_filename(qc_example)         # nothing is written: only a string is produced
print(f"Generated filename: {filename}")
print("\nNaming best practices:")
print("  ✓ Use descriptive names")
print("  ✓ Include qubit count")
print("  ✓ Version your circuits")
print("  ✓ Use lowercase and underscores")
print("  ✓ Add .qpy extension")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `generate_circuit_filename(circuit, version)` | derives the file name from the circuit's own attributes (name, qubit count) so that the name can never disagree with the content |
| `.lower().replace(' ', '_')` | string methods chained left to right: `'Grover Oracle'` → `'grover_oracle'` |

</details>


### 2. Error Handling

In [ ]:
# Robust save/load functions
def safe_save(circuit, filename):
    """Safely save circuit with error handling."""
    try:                                               # try the risky operation ...
        with open(filename, 'wb') as file:
            qpy.dump(circuit, file)
        print(f"✓ Successfully saved: {filename}")
        return True
    except Exception as e:                             # ... and report instead of crashing if anything goes wrong
        print(f"✗ Error saving {filename}: {e}")
        return False

def safe_load(filename):
    """Safely load circuit with error handling."""
    try:
        with open(filename, 'rb') as file:
            circuits = qpy.load(file)
        print(f"✓ Successfully loaded: {filename}")
        return circuits
    except FileNotFoundError:                          # the most common failure gets its own, specific message
        print(f"✗ File not found: {filename}")
        return None
    except Exception as e:                             # any other error (corrupted file, wrong format, ...)
        print(f"✗ Error loading {filename}: {e}")
        return None

# Test
test_qc = QuantumCircuit(2)
test_qc.h(0)
safe_save(test_qc, 'test_safe.qpy')
loaded = safe_load('test_safe.qpy')
loaded_missing = safe_load('nonexistent.qpy')           # returns None and prints the message: no crash


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `try: ... except FileNotFoundError: ... except Exception as e:` | exception handling from specific to generic: Python runs the first matching `except`; the specific one gives a clear message, the generic one catches everything else |
| returning `True`/`False`/`None` | the caller can test the outcome (`if circuits is None: ...`) instead of wrapping every call in its own try/except |
| `safe_load('nonexistent.qpy')` | the deliberate failure: the ✗ line is the expected output, not an error of the notebook |

</details>


### 3. Version Control Integration

In [ ]:
# Tips for Git integration                      (this cell only prints advice: nothing is written to disk)
print("Git Best Practices for QPY Files:")
print("="*60)                                      # a 60-character ruler: string repetition
print("\n1. .gitignore recommendations:")
print("   # Ignore temporary circuit files")
print("   *_temp.qpy")
print("   *_scratch.qpy")
print("")
print("2. Commit messages:")
print("   ✓ 'Add: Bell state circuit for teleportation'")
print("   ✓ 'Update: Increase QFT depth to 5 qubits'")
print("   ✓ 'Fix: Correct phase in Grover oracle'")
print("")
print("3. Directory structure:")
print("   circuits/")
print("     ├── algorithms/    (versioned algorithms)")
print("     ├── benchmarks/    (standard test circuits)")
print("     └── experiments/   (research circuits)")
print("")
print("4. Documentation:")
print("   - Include README in circuits/ directory")
print("   - Document circuit purpose and parameters")
print("   - Note Qiskit version used")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `print("="*60)` | string × integer repeats the string: a quick separator line |
| QPY and Git | QPY files are binary, so Git stores whole-file versions (no line diffs): keep them small, name them well, and put the *description* of a change in the commit message and in `metadata` |

</details>


## 🧹 Cleanup

In [ ]:
# Clean up created files — run for real, so that the course folder stays clean after the notebook
import shutil                                      # high-level file operations: rmtree removes a whole directory tree

def cleanup_demo_files():
    """Remove demo files created in this notebook."""
    # Remove individual files
    files_to_remove = [
        'bell_state.qpy',
        'circuit_library.qpy',
        'circuit_with_metadata.qpy',
        'test_safe.qpy'
    ]

    for file in files_to_remove:
        try:
            os.remove(file)                        # delete one file
            print(f"✓ Removed: {file}")
        except FileNotFoundError:                  # already gone (e.g. the cell was run twice): ignore
            pass

    # Remove circuits directory
    try:
        shutil.rmtree('circuits')                  # delete the folder and everything inside it
        print("✓ Removed: circuits/ directory")
    except FileNotFoundError:
        pass

cleanup_demo_files()                               # comment this line out if you want to keep the files and inspect them
print("Demo files removed — re-run the notebook from the top to create them again.")


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `cleanup_demo_files()` | runs the cleanup (the original left it commented out): every file and folder written by this notebook is removed, so the course directory is exactly as before. Comment the call out to keep the files |
| `os.remove` vs `shutil.rmtree` | `os.remove` deletes one file; `shutil.rmtree` deletes a directory recursively — powerful, so always pass a path you built yourself |
| `except FileNotFoundError: pass` | idempotence: re-running the cell must not fail because the files are already gone |

</details>


## 🎓 Exercises

### Exercise 1: Save Your Own Circuit (Easy)
Create a 3-qubit GHZ state circuit, add metadata with your name and date, then save and reload it.

```python
# Your solution here
```

### Exercise 2: Circuit Library (Medium)
Create a library of 5 different algorithm circuits (your choice), save them to an organized directory structure, and create a function to load any circuit by name.

### Exercise 3: Version Management (Hard)
Implement a circuit versioning system that:
1. Saves circuits with version numbers
2. Tracks changes in metadata
3. Can rollback to previous versions
4. Generates a version history report

## 🔬 In-depth analysis

All three tasks work inside a `tempfile.TemporaryDirectory()` so that nothing is left on disk.

1. **Exercise 1 of the author, solved.** Build a 3-qubit GHZ circuit, attach `metadata` with your name and today's date, save it with `qpy.dump`, reload it and check that the circuit and the metadata survived.
2. **A library with named access.** Write `load_by_name(path, name)` that opens a multi-circuit QPY file and returns the circuit called `name` (or `None`). Test it on the four circuits of this notebook saved into one file.
3. **An exact W state.** The `W_State` circuit above is not a W state. Prepare W = (\|001⟩ + \|010⟩ + \|100⟩)/√3 with `x(0)`, `cry(2·arccos(1/√3), 0, 1)`, `cx(1, 0)`, `cry(π/2, 1, 2)`, `cx(2, 1)`, save and reload it, and verify with `Statevector(...).probabilities_dict()` that the three probabilities are 1/3.

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [ ]:
# 🔬 In-depth analysis — your code here



In [ ]:
# Reference solution
import tempfile, datetime
from pathlib import Path                                   # object-oriented paths: tmp / 'file.qpy' joins with the right separator
from qiskit.quantum_info import Statevector

with tempfile.TemporaryDirectory(prefix='qpy_demo_') as tmp:   # a scratch folder, deleted automatically at the end of the with-block
    tmp = Path(tmp)

    # (1) Exercise 1: GHZ with metadata, saved and reloaded
    ghz = QuantumCircuit(3, name='GHZ_documented')
    ghz.h(0); ghz.cx(0, 1); ghz.cx(0, 2)
    ghz.metadata = {'author': 'your name', 'date': datetime.date.today().isoformat(), 'state': '(|000> + |111>)/sqrt2'}
    with open(tmp / 'ghz.qpy', 'wb') as f: qpy.dump(ghz, f)
    with open(tmp / 'ghz.qpy', 'rb') as f: ghz_back = qpy.load(f)[0]
    print(f"(1) reloaded {ghz_back.name}: equal to the original: {ghz_back == ghz}; metadata: {ghz_back.metadata}")

    # (2) a library file and access by name
    def load_by_name(path, name):
        """Return the circuit called `name` from a multi-circuit QPY file, or None."""
        with open(path, 'rb') as f:
            for circ in qpy.load(f):                       # linear scan: fine for a library of tens of circuits
                if circ.name == name:
                    return circ
        return None
    with open(tmp / 'library.qpy', 'wb') as f: qpy.dump(circuits, f)   # the four circuits of this notebook
    found = load_by_name(tmp / 'library.qpy', 'QFT_3')
    missing = load_by_name(tmp / 'library.qpy', 'Grover')
    print(f"(2) load_by_name('QFT_3') -> {found.name} ({found.num_qubits} qubits, depth {found.decompose().depth()} once decomposed); "
          f"load_by_name('Grover') -> {missing}")

    # (3) an exact W state, saved and reloaded
    theta = 2 * np.arccos(1 / np.sqrt(3))                  # cos(theta/2) = 1/sqrt3
    w = QuantumCircuit(3, name='W_exact')
    w.x(0)                                                 # |001>
    w.cry(theta, 0, 1); w.cx(1, 0)                         # move amplitude sqrt(2/3) from qubit 0 to qubit 1: (|001> + sqrt2 |010>)/sqrt3
    w.cry(np.pi / 2, 1, 2); w.cx(2, 1)                     # split qubit 1's share equally with qubit 2: (|001> + |010> + |100>)/sqrt3
    with open(tmp / 'w.qpy', 'wb') as f: qpy.dump(w, f)
    with open(tmp / 'w.qpy', 'rb') as f: w_back = qpy.load(f)[0]
    probs = {str(k): round(float(v), 4) for k, v in Statevector(w_back).probabilities_dict().items()}   # dict comprehension: tidy keys and floats
    print(f"(3) probabilities of the reloaded W circuit: {probs}   (each 1/3 = 0.3333)")
    display(draw(w)); print("↑ the exact W-state circuit: X, two controlled-Ry rotations and two CNOTs that 'hand back' the control")

    print(f"scratch files written: {sorted(p.name for p in tmp.iterdir())} — deleted when the with-block ends")
print("scratch folder still exists:", tmp.exists())        # False: TemporaryDirectory cleaned up after itself


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `tempfile.TemporaryDirectory()` as a context manager | the right tool for files you do not want to keep: a unique folder is created, and removed — with its contents — when the block ends, even after an error |
| `Path(tmp) / 'ghz.qpy'` | `pathlib` builds paths with `/`; `open()` accepts `Path` objects, so no string gluing is needed |
| `load_by_name` | a linear search over `qpy.load(f)`: names are the robust key for a library (file order is not); returning `None` lets the caller test `if circ is None` |
| `cry(θ, control, target)` + `cx(target, control)` | the W-state trick: a controlled rotation moves part of the amplitude to a new qubit, the CNOT then switches the old qubit off in that branch so that exactly one qubit is 1 in every term |
| `Statevector(w_back).probabilities_dict()` | the exact probabilities of the *reloaded* circuit: 0.3333 on `001`, `010`, `100` — proof that QPY preserved the rotation angles to full precision |

</details>


<details open><summary><b>📐 Working it out</b></summary>

Write $c=\cos(\theta/2)$, $s=\sin(\theta/2)$ with $\theta = 2\arccos(1/\sqrt3)$, so $c=1/\sqrt3$ and $s=\sqrt{2/3}$. Kets are $|q_2q_1q_0\rangle$.

1. `x(0)`: $|001\rangle$.
2. `cry(θ, 0, 1)` (control $q_0=1$): $|001\rangle \to c\,|001\rangle + s\,|011\rangle$.
3. `cx(1, 0)`: in the branch with $q_1=1$ the control fires and $q_0\to0$: $c\,|001\rangle + s\,|010\rangle$.
4. `cry(π/2, 1, 2)` (control $q_1=1$): $|010\rangle\to\frac{1}{\sqrt2}(|010\rangle+|110\rangle)$, giving $c\,|001\rangle+\frac{s}{\sqrt2}\,|010\rangle+\frac{s}{\sqrt2}\,|110\rangle$.
5. `cx(2, 1)`: where $q_2=1$, $q_1\to0$: $c\,|001\rangle+\frac{s}{\sqrt2}\,|010\rangle+\frac{s}{\sqrt2}\,|100\rangle$.

With $c=1/\sqrt3$ and $s/\sqrt2=\sqrt{2/3}/\sqrt2=1/\sqrt3$ all three amplitudes are $1/\sqrt3$: probabilities $1/3 = 0.3333$ each, as printed in (3).

</details>


## 📖 References

### Documentation

1. **Qiskit QPY Format**:  
   https://docs.quantum.ibm.com/api/qiskit/qpy

2. **Circuit Serialization Guide**:  
   https://docs.quantum.ibm.com/guides/save-circuits

### Best Practices

- Always use QPY format (not pickle) for forward compatibility
- Include metadata for traceability
- Use version control for circuit evolution
- Organize circuits by purpose/algorithm
- Document circuit parameters and usage

---

**Course**: Quantum Computing Laboratory  
**License**: MIT  
**Version**: 2.2.0  
**Last Updated**: November 2025